# Silicon Poles on democracy: feasibility test (NLP4CA proposal 1)

Do LLM personas built from real Polish survey respondents describe **democracy** the way those respondents did,
and does the **prompt language** (Polish vs English) change that?

This notebook runs everything on CU Alpine through Open OnDemand, no terminal needed:
starts its own Ollama server on the GPU, downloads the models, generates silicon answers,
codes all answers with the Democracy Tree scheme, embeds them, and runs the analysis.

**Data:** LES Democracy Survey 2020, CC0 replication package of Dahlberg et al. (2026, *Political Analysis*),
Dürlich et al. (2026) Harvard Dataverse doi:10.7910/DVN/XOWF9C. Derived inputs are in `data/inputs/` (built by `src/prep.py`).

### How to launch on Alpine (Open OnDemand)
1. https://ondemand.rc.colorado.edu → **Interactive Apps → Jupyter Session**
2. Configuration type **Custom configuration**: Cluster `alpine`, Account `ucb-general`, Partition `aa100`,
   Account: your allocation (`ucb757_asc1`) or `ucb-general`. Any of these GPUs works (the A100 pool is often the busiest,
   so queue two or three at once and delete the others as soon as one starts; never Run All in two sessions at the same time):
   - Partition `aa100`, QoS `gpu-normal`, gres `gpu:a100-40gb:1`, Time `2`, cores `8`
   - Partition `artxpro6000`, QoS `gpu-normal`, gres `gpu:rtx_pro_6000_2g.48gb:1` (or `gpu:rtx_pro_6000_1g.24gb:1`), Time `2`, cores `8`
   - Partition `ah200`, QoS `gpu-normal`, gres `gpu:h200_2g.35gb:1`, Time `2`, cores `8`
   - Partition `al40`, QoS `gpu-normal`, gres `gpu:l40:1`, Time `2`, cores `8`
   - testing slice (usually starts at once, 1 h max, the notebook resumes across sessions): Partition `aa100`, QoS `gpu-testing`, gres `gpu:a100_3g.20gb:1`, Time `1`, cores `10`
3. **Connect to Jupyter**, then File → Open from Path → `/projects/<you>`; clone this repo there (cell below) and open this notebook.
4. Run with `SMOKE = True` first (3 items per stage), then set `SMOKE = False` and **Run All** again.
   Every stage checkpoints to `outputs/`: if the session ends, just run again and it continues.

In [1]:
# (only once) clone the repo into /projects/$USER, then open notebooks/feasibility_alpine.ipynb from the file browser
# !cd /projects/$USER && git clone https://github.com/AntoniCzolgowski/silicon-democracy-nlp4ca.git
# update to the latest version later (keeps outputs/; resets this notebook, so set SMOKE again afterwards):
# !cd /projects/$USER/silicon-democracy-nlp4ca && git fetch -q && git reset --hard origin/main

## 1. Settings

In [2]:
SMOKE = False            # True: 3 items per stage (end-to-end check). Then set False and Run All.

GEN_MODELS = {'gemma': 'gemma3:12b',                                   # global open model
              'bielik': 'SpeakLeash/bielik-11b-v3.0-instruct:Q4_K_M'}  # Polish-native open model
CODER_MODEL = 'gemma3:27b'       # larger model as the Democracy Tree coder (validated against human codes)
EMBED_MODEL = 'embeddinggemma'   # multilingual embeddings for the homogeneity measure
TEMPERATURE = 1.0                # sampling temperature for the silicon respondents

import os, sys, json, time, socket, shutil, hashlib, subprocess, threading, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

ROOT = Path.cwd()
while not (ROOT / 'data' / 'inputs').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'inputs').exists(), 'Open this notebook from inside the cloned repo'
IN, OUT = ROOT / 'data' / 'inputs', ROOT / 'outputs'
OUT.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT / 'src'))
USER = os.environ.get('USER', 'me')
OLLAMA_MODELS = os.environ.get('OLLAMA_MODELS_DIR', f'/projects/{USER}/ollama_models')
LIMIT = 3 if SMOKE else None
print('repo:', ROOT, '| smoke test:', SMOKE, '| models dir:', OLLAMA_MODELS)

repo: /projects/ancz7294/silicon-democracy-nlp4ca | smoke test: False | models dir: /projects/ancz7294/ollama_models


## 2. GPU and Ollama server
The server is started from this notebook with CURC's `ollama` module and our own model folder in `/projects` (~33 GB).

In [3]:
try:
    gpu = subprocess.run(['nvidia-smi', '-L'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True).stdout
except FileNotFoundError:
    gpu = 'NO GPU'
    if not os.environ.get('OLLAMA_URL'):
        raise RuntimeError('No GPU in this session: relaunch the Jupyter Session on a GPU partition with a gres (see top of notebook)')
print(gpu)
import re
mig = re.findall(r'MIG\s+\d+g\.(\d+)gb', gpu)                   # e.g. 'MIG 3g.20gb' -> 20 GB slice
if mig:
    VRAM = int(mig[0])
else:                                                          # full GPU: ask nvidia-smi for its memory
    try:
        q = subprocess.run(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
                           stdout=subprocess.PIPE, universal_newlines=True).stdout.split()
        VRAM = int(int(q[0]) / 1024)
    except Exception:
        VRAM = 40
# the 27B coder needs ~17 GB + ~2 GB per parallel request
PAR, GEN_WORKERS, CODE_WORKERS = (1, 2, 1) if VRAM <= 20 else (2, 3, 2) if VRAM <= 30 else (4, 4, 3)
print(f'GPU memory for us: ~{VRAM} GB ({"MIG slice" if mig else "full GPU"}) | parallel requests: {PAR}')

GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-455cc7c5-1cab-29c6-3f7b-68600e0f5062)
GPU 1: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-b7e03695-8c12-68ba-f353-b0e61367877d)
GPU 2: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-d5fb7d85-074f-f830-23e1-b3e32c1a6980)
GPU 3: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-742d5daf-8a6d-2bcd-d58a-563c1a3a69b2)
  MIG 2g.48gb     Device  0: (UUID: MIG-7a6b1b03-0955-5be4-860d-7f568867a1c9)

GPU memory for us: ~48 GB (MIG slice) | parallel requests: 4


In [4]:
def free_port():
    s = socket.socket(); s.bind(('127.0.0.1', 0)); p = s.getsockname()[1]; s.close(); return p

def module_env():
    # environment that CURC's `module load ollama` sets up (PATH, LD_LIBRARY_PATH, ...)
    cmd = f'export OLLAMA_MODELS={OLLAMA_MODELS}; module load ollama >/dev/null 2>&1; env -0'
    raw = subprocess.run(['bash', '-lc', cmd], stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=180).stdout
    return dict(kv.split('=', 1) for kv in raw.decode(errors='ignore').split('\0') if '=' in kv and not kv.startswith('BASH_FUNC'))

def api(path, payload=None, timeout=900):
    data = None if payload is None else json.dumps(payload).encode()
    req = urllib.request.Request(BASE + path, data=data, headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

if os.environ.get('OLLAMA_URL'):                     # use an already running server (testing)
    BASE, server = os.environ['OLLAMA_URL'], None
else:
    env = module_env()
    exe = shutil.which('ollama', path=env.get('PATH', ''))
    assert exe, 'CURC ollama module not found: run this on an Alpine compute node (Jupyter Session on a GPU)'
    subprocess.run(['pkill', '-u', USER, '-f', 'ollama serve'])     # stop servers left by the module or earlier runs
    time.sleep(2)
    port = free_port()
    env.update(OLLAMA_HOST=f'127.0.0.1:{port}', OLLAMA_MODELS=OLLAMA_MODELS, OLLAMA_NUM_PARALLEL=str(PAR),
               OLLAMA_MAX_LOADED_MODELS='1', OLLAMA_KEEP_ALIVE='30m')
    Path(OLLAMA_MODELS).mkdir(parents=True, exist_ok=True)
    server = subprocess.Popen([exe, 'serve'], env=env, stdout=open(OUT / 'ollama_server.log', 'a'), stderr=subprocess.STDOUT)
    BASE = f'http://127.0.0.1:{port}'
for _ in range(60):
    try:
        print('Ollama', api('/api/version')['version'], 'at', BASE); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama did not start; see outputs/ollama_server.log')

Ollama 0.33.2 at http://127.0.0.1:49335


## 3. Download models (first time only, 5-15 min)

In [5]:
def pull(model):
    have = {m['name'] for m in api('/api/tags').get('models', [])}
    if model in have or f'{model}:latest' in have:
        print('have', model); return
    req = urllib.request.Request(BASE + '/api/pull', data=json.dumps({'model': model, 'stream': True}).encode(),
                                 headers={'Content-Type': 'application/json'})
    last = -10
    with urllib.request.urlopen(req, timeout=7200) as r:
        for line in r:
            ev = json.loads(line)
            if 'error' in ev: raise RuntimeError(f"{model}: {ev['error']}")
            if ev.get('total') and ev.get('completed'):
                pct = 100 * ev['completed'] / ev['total']
                if pct - last >= 10: print(f'  {model}: {pct:.0f}%'); last = pct
    print('pulled', model)

for m in [*GEN_MODELS.values(), CODER_MODEL, EMBED_MODEL]:
    pull(m)
print([m['name'] for m in api('/api/tags')['models']])

have gemma3:12b
have SpeakLeash/bielik-11b-v3.0-instruct:Q4_K_M
have gemma3:27b
have embeddinggemma
['embeddinggemma:latest', 'gemma3:27b', 'SpeakLeash/bielik-11b-v3.0-instruct:Q4_K_M', 'gemma3:12b', 'qwen2.5:14b', 'qwen2.5:0.5b']


## 4. Inputs and prompts
Persona prompts carry the respondent's gender, age, education, employment, political interest and three immigration attitudes; the question is the exact LES 2020 wording. No instruction about answer length is given, so length is an outcome.

In [6]:
from personas import persona_messages, CODER_SYSTEM

def read_jsonl(p):
    p = Path(p); return [json.loads(l) for l in open(p, encoding='utf-8')] if p.exists() else []

personas = read_jsonl(IN / 'personas.jsonl')
print(len(personas), 'personas;', len(read_jsonl(IN / 'real_pl.jsonl')), 'real PL;', len(read_jsonl(IN / 'real_us.jsonl')), 'real US;',
      len(read_jsonl(IN / 'validation.jsonl')), 'human-coded validation answers')
for lang in ('pl', 'en'):
    m = persona_messages(personas[0], lang)
    print(f'\n[{lang}] SYSTEM: {m[0]["content"]}\n[{lang}] USER:   {m[1]["content"]}')
print('\nreal answer of this person:', personas[0]['real_q4'])

150 personas; 300 real PL; 150 real US; 200 human-coded validation answers

[pl] SYSTEM: Bierzesz udział w internetowej ankiecie prowadzonej w Polsce w 2020 roku. Odpowiadaj tak, jak odpowiedziałaby poniższa osoba. O Tobie: jesteś mężczyzną w wieku 58 lat i mieszkasz w Polsce. Masz wyższe wykształcenie. Pracujesz zarobkowo. Ani się zgadzasz, ani nie zgadzasz ze stwierdzeniem, że imigranci są na ogół dobrzy dla polskiej gospodarki. Nie zgadzasz się ze stwierdzeniem, że imigranci zagrażają polskiej kulturze. Ani się zgadzasz, ani nie zgadzasz ze stwierdzeniem, że imigranci zwiększają przestępczość. Wpisz odpowiedź w pole tekstowe ankiety tak, jak zrobiłaby to ta osoba. Podaj wyłącznie samą odpowiedź.
[pl] USER:   Co przychodzi Panu do głowy, kiedy słyszy Pan słowo „demokracja”? Proszę to zapisać w kilku linijkach.

[en] SYSTEM: You are taking part in an online survey conducted in Poland in 2020. Answer as the following person would. About you: you are a 58-year-old man living in Poland. 

## 5. Helpers (parallel, checkpointed)

In [7]:
LOCK = threading.Lock()
def append(path, obj):
    with LOCK, open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(obj, ensure_ascii=False) + '\n')

def run_pool(jobs, fn, workers, label):
    t0, done, errors = time.time(), 0, 0
    with ThreadPoolExecutor(workers) as ex:
        futs = [ex.submit(fn, j) for j in jobs]
        for f in as_completed(futs):
            done += 1
            try: f.result()
            except Exception as e:
                errors += 1; print('ERROR', repr(e)[:200])
            if done % 25 == 0 or done == len(jobs):
                print(f'[{label}] {done}/{len(jobs)}  {time.time() - t0:.0f}s  errors={errors}', flush=True)

def seed_for(*parts):
    return int(hashlib.md5('|'.join(map(str, parts)).encode()).hexdigest()[:8], 16)

## 6. Generate silicon answers (2 models × 2 prompt languages × 150 personas)

In [8]:
SIL = OUT / 'silicon.jsonl'
for short, model in GEN_MODELS.items():
    done = {(r['model'], r['lang'], r['respondent_id']) for r in read_jsonl(SIL)}
    jobs = [(p, lang) for p in personas[:LIMIT] for lang in ('pl', 'en') if (model, lang, p['respondent_id']) not in done]
    print(model, '->', len(jobs), 'to generate')

    def one(job, model=model, short=short):
        p, lang = job
        msgs = persona_messages(p, lang); t = time.time()
        r = api('/api/chat', {'model': model, 'messages': msgs, 'stream': False,
                              'options': {'temperature': TEMPERATURE, 'top_p': 0.95, 'num_predict': 400,
                                          'seed': seed_for(model, lang, p['respondent_id'])}})
        append(SIL, {'item_id': f"sil_{short}_{lang}_{p['respondent_id']}", 'corpus': f'silicon_{short}_{lang}',
                     'model': model, 'lang': lang, 'respondent_id': p['respondent_id'], 'edu': p['edu'],
                     'gender': p['gender'], 'age': p['age'], 'text': r['message']['content'].strip(),
                     'temperature': TEMPERATURE, 'seconds': round(time.time() - t, 2)})
    run_pool(jobs, one, GEN_WORKERS, short)

for r in read_jsonl(SIL)[:4]:
    print(f"\n{r['corpus']} | {r['gender']} {r['age']} {r['edu']}:\n  {r['text'][:400]}")

gemma3:12b -> 294 to generate
[gemma] 25/294  136s  errors=0
[gemma] 50/294  164s  errors=0
[gemma] 75/294  196s  errors=0
[gemma] 100/294  223s  errors=0
[gemma] 125/294  254s  errors=0
[gemma] 150/294  282s  errors=0
[gemma] 175/294  313s  errors=0
[gemma] 200/294  341s  errors=0
[gemma] 225/294  371s  errors=0
[gemma] 250/294  403s  errors=0
[gemma] 275/294  432s  errors=0
[gemma] 294/294  451s  errors=0
SpeakLeash/bielik-11b-v3.0-instruct:Q4_K_M -> 294 to generate
[bielik] 25/294  102s  errors=0
[bielik] 50/294  140s  errors=0
[bielik] 75/294  182s  errors=0
[bielik] 100/294  222s  errors=0
[bielik] 125/294  262s  errors=0
[bielik] 150/294  301s  errors=0
[bielik] 175/294  339s  errors=0
[bielik] 200/294  380s  errors=0
[bielik] 225/294  418s  errors=0
[bielik] 250/294  455s  errors=0
[bielik] 275/294  494s  errors=0
[bielik] 294/294  521s  errors=0

silicon_gemma_en | male 58 high:
  Democracy… well, it's complicated. It’s the system we have, and it's better than the alternatives 

## 7. Code every answer with the Democracy Tree (validation set in Polish and English, real PL, real US, all silicon)

In [9]:
CODED = OUT / 'coded.jsonl'

def parse_json(s):
    s = s.strip()
    i, j = s.find('{'), s.rfind('}')
    return json.loads(s[i:j + 1])

items = []
for v in read_jsonl(IN / 'validation.jsonl'):
    items.append({'item_id': f"val_pl_{v['ann_id']}", 'corpus': 'validation_pl', 'text': v['text_pl']})
    items.append({'item_id': f"val_en_{v['ann_id']}", 'corpus': 'validation_en', 'text': v['text_en']})
items += read_jsonl(IN / 'real_pl.jsonl') + read_jsonl(IN / 'real_us.jsonl') + read_jsonl(SIL)
if LIMIT:
    items = items[:LIMIT]
done = {(r['item_id'], r['coder']) for r in read_jsonl(CODED)}
jobs = [it for it in items if (it['item_id'], CODER_MODEL) not in done]
print(len(jobs), 'to code with', CODER_MODEL)

def code_one(it):
    r = api('/api/chat', {'model': CODER_MODEL, 'stream': False, 'format': 'json',
                          'messages': [{'role': 'system', 'content': CODER_SYSTEM},
                                       {'role': 'user', 'content': 'Answer to code:\n<answer>' + it['text'] + '</answer>'}],
                          'options': {'temperature': 0, 'num_predict': 120, 'num_ctx': 4096}})
    raw = r['message']['content']
    try: lab = parse_json(raw)
    except Exception: lab = {'parse_error': True}
    append(CODED, {'item_id': it['item_id'], 'corpus': it['corpus'], 'coder': CODER_MODEL, 'labels': lab, 'raw': raw})

run_pool(jobs, code_one, CODE_WORKERS, 'code')
print(read_jsonl(CODED)[:2])

1447 to code with gemma3:27b
[code] 25/1447  146s  errors=0
[code] 50/1447  180s  errors=0
[code] 75/1447  212s  errors=0
[code] 100/1447  245s  errors=0
[code] 125/1447  273s  errors=0
[code] 150/1447  303s  errors=0
[code] 175/1447  336s  errors=0
[code] 200/1447  365s  errors=0
[code] 225/1447  393s  errors=0
[code] 250/1447  425s  errors=0
[code] 275/1447  456s  errors=0
[code] 300/1447  484s  errors=0
[code] 325/1447  518s  errors=0
[code] 350/1447  546s  errors=0
[code] 375/1447  575s  errors=0
[code] 400/1447  605s  errors=0
[code] 425/1447  633s  errors=0
[code] 450/1447  661s  errors=0
[code] 475/1447  693s  errors=0
[code] 500/1447  722s  errors=0
[code] 525/1447  754s  errors=0
[code] 550/1447  789s  errors=0
[code] 575/1447  816s  errors=0
[code] 600/1447  844s  errors=0
[code] 625/1447  877s  errors=0
[code] 650/1447  904s  errors=0
[code] 675/1447  930s  errors=0
[code] 700/1447  961s  errors=0
[code] 725/1447  988s  errors=0
[code] 750/1447  1015s  errors=0
[code] 775/14

## 8. Embeddings (homogeneity measure)

In [10]:
EMB = OUT / 'embeddings.jsonl'
emb_items = read_jsonl(IN / 'real_pl.jsonl') + read_jsonl(IN / 'real_us.jsonl') + read_jsonl(SIL)
if LIMIT: emb_items = emb_items[:LIMIT]
done = {r['item_id'] for r in read_jsonl(EMB)}
todo = [it for it in emb_items if it['item_id'] not in done]
print(len(todo), 'to embed')
for k in range(0, len(todo), 32):
    batch = todo[k:k + 32]
    r = api('/api/embed', {'model': EMBED_MODEL, 'input': [b['text'] for b in batch]})
    for b, e in zip(batch, r['embeddings']):
        append(EMB, {'item_id': b['item_id'], 'corpus': b['corpus'], 'model': EMBED_MODEL, 'vec': [round(x, 5) for x in e]})
print('embeddings:', len(read_jsonl(EMB)))

1047 to embed
embeddings: 1050


## 9. Run manifest (for reproducibility)

In [11]:
manifest = {'time': time.strftime('%Y-%m-%d %H:%M:%S'), 'smoke': SMOKE, 'gpu': gpu.strip(),
            'ollama_version': api('/api/version')['version'], 'temperature': TEMPERATURE,
            'models': [{**{k: m.get(k) for k in ('name', 'digest', 'size')}, 'details': m.get('details')} for m in api('/api/tags')['models']],
            'counts': {p.name: sum(1 for _ in open(p, encoding='utf-8')) for p in OUT.glob('*.jsonl')}}
json.dump(manifest, open(OUT / 'manifest.json', 'w'), indent=1)
manifest['counts']

{'coded.jsonl': 1450, 'silicon.jsonl': 600, 'embeddings.jsonl': 1050}

## 10. Analysis
Coder validation against the human codes, corpus descriptives, JSD of Democracy Tree profiles with a split-half baseline, the paired prompt-language effect (H2), education gap, individual-level agreement, lexical markers, type diversity and embedding homogeneity.

In [12]:
from IPython.display import Markdown, display
if SMOKE:
    print('Smoke test finished. Set SMOKE = False and Run All for the full run; the analysis runs on the full data.')
else:
    subprocess.run([sys.executable, str(ROOT / 'src' / 'analyze.py')], check=True, stdout=subprocess.DEVNULL)
    display(Markdown((OUT / 'results.md').read_text(encoding='utf-8')))

# Feasibility results

coded items: 1450, parse errors: 0

## Coder validation (Polish Q4, n=200)
| domain | gold rate | coder rate | F1 | kappa coder-gold | kappa human-human | kappa PL vs EN coding |
|---|---|---|---|---|---|---|
| governance | 0.51 | 0.55 | 0.89 | 0.76 | 0.74 | 0.84 |
| values | 0.56 | 0.59 | 0.91 | 0.79 | 0.83 | 0.85 |
| outcome | 0.22 | 0.17 | 0.63 | 0.54 | 0.60 | 0.79 |
| nonresponse | 0.09 | 0.04 | 0.64 | 0.62 | 0.86 | 0.93 |
macro-F1 0.77; valence acc 0.88, kappa 0.76 (human-human 0.60)

## Corpora
| corpus | n | words mean | median | <=3 words | gov | val | out | nonresp | positive | negative |
|---|---|---|---|---|---|---|---|---|---|---|
| real_pl | 150 | 7.09 | 5.00 | 0.40 | 0.51 | 0.57 | 0.21 | 0.05 | 0.61 | 0.11 |
| real_pl_all300 | 300 | 7.18 | 5.00 | 0.38 | 0.56 | 0.53 | 0.18 | 0.04 | 0.61 | 0.09 |
| silicon_bielik_en | 150 | 67.59 | 66.00 | 0.00 | 0.96 | 0.79 | 0.89 | 0.01 | 0.37 | 0.00 |
| silicon_bielik_pl | 150 | 64.90 | 65.00 | 0.00 | 1.00 | 0.83 | 0.95 | 0.00 | 0.31 | 0.03 |
| silicon_gemma_en | 150 | 73.81 | 74.00 | 0.00 | 0.96 | 0.71 | 0.93 | 0.00 | 0.16 | 0.15 |
| silicon_gemma_pl | 150 | 54.10 | 52.00 | 0.00 | 0.90 | 0.86 | 0.89 | 0.01 | 0.27 | 0.07 |
| real_us | 150 | 8.75 | 5.00 | 0.37 | 0.51 | 0.49 | 0.27 | 0.09 | 0.57 | 0.11 |

split-half JSD within real PL: mean 0.0064, 95th pct 0.0149

## JSD of domain profiles (bootstrap 95% CI)
- silicon_bielik_en vs_real_pl: 0.0443 [0.0261, 0.0744]
- silicon_bielik_en vs_real_us: 0.0457 [0.0242, 0.0768]
- silicon_bielik_pl vs_real_pl: 0.0548 [0.0345, 0.0819]
- silicon_bielik_pl vs_real_us: 0.0579 [0.0359, 0.0863]
- silicon_gemma_en vs_real_pl: 0.0622 [0.0399, 0.0929]
- silicon_gemma_en vs_real_us: 0.0627 [0.0396, 0.0885]
- silicon_gemma_pl vs_real_pl: 0.0457 [0.0266, 0.0736]
- silicon_gemma_pl vs_real_us: 0.0497 [0.0278, 0.0773]
- real_us vs_real_pl: 0.0075 [0.0026, 0.0335]

## H2 language effect: JSD(EN prompt) - JSD(PL prompt), paired bootstrap
- bielik_real_pl: -0.0104 [-0.0200, +0.0028] (n=150)
- bielik_real_us: -0.0122 [-0.0255, +0.0014] (n=150)
- gemma_real_pl: +0.0165 [+0.0050, +0.0287] (n=150)
- gemma_real_us: +0.0130 [+0.0026, +0.0227] (n=150)

## Education gap (high - low)
- real_pl: governance -0.04, values +0.08, outcome -0.01, nonresponse -0.03, other -0.01
- real_pl_all300: governance -0.08, values +0.10, outcome +0.06, nonresponse -0.01, other -0.01
- silicon_bielik_en: governance -0.04, values +0.13, outcome -0.05, nonresponse -0.02, other -0.02
- silicon_bielik_pl: governance +0.00, values +0.25, outcome -0.05, nonresponse +0.00, other +0.00
- silicon_gemma_en: governance +0.07, values +0.33, outcome +0.03, nonresponse +0.00, other +0.00
- silicon_gemma_pl: governance -0.08, values +0.27, outcome -0.12, nonresponse -0.02, other +0.00

## Individual-level agreement with the same person (kappa)
- silicon_bielik_en: governance acc 0.53 k 0.06, values acc 0.53 k -0.01, outcome acc 0.26 k -0.03, nonresponse acc 0.95 k -0.01
- silicon_bielik_pl: governance acc 0.51 k 0.00, values acc 0.57 k 0.06, outcome acc 0.25 k 0.01, nonresponse acc 0.95 k 0.00
- silicon_gemma_en: governance acc 0.51 k 0.00, values acc 0.55 k 0.04, outcome acc 0.25 k 0.00, nonresponse acc 0.95 k 0.00
- silicon_gemma_pl: governance acc 0.53 k 0.04, values acc 0.55 k -0.00, outcome acc 0.27 k -0.01, nonresponse acc 0.95 k -0.01

## Lexical markers (share of answers)
- real_pl: elections/voting 0.23, freedom 0.37, equality 0.12, law/rights 0.24, majority/people 0.29, corruption/negative 0.01
- real_pl_all300: elections/voting 0.25, freedom 0.33, equality 0.13, law/rights 0.25, majority/people 0.32, corruption/negative 0.01
- silicon_bielik_en: elections/voting 0.40, freedom 0.45, equality 0.33, law/rights 0.57, majority/people 0.65, corruption/negative 0.01
- silicon_bielik_pl: elections/voting 0.97, freedom 0.66, equality 0.40, law/rights 0.92, majority/people 0.95, corruption/negative 0.03
- silicon_gemma_en: elections/voting 0.49, freedom 0.20, equality 0.00, law/rights 0.25, majority/people 0.54, corruption/negative 0.01
- silicon_gemma_pl: elections/voting 0.70, freedom 0.61, equality 0.05, law/rights 0.59, majority/people 0.69, corruption/negative 0.03
- real_us: elections/voting 0.17, freedom 0.23, equality 0.06, law/rights 0.09, majority/people 0.21, corruption/negative 0.01
- real_pl_machine_translated_en: elections/voting 0.13, freedom 0.38, equality 0.16, law/rights 0.16, majority/people 0.29, corruption/negative 0.01

## Type diversity at equal token budget
- real_pl: 485.0 types / 1071 tokens
- silicon_bielik_pl: 360.7 types / 1071 tokens
- silicon_gemma_pl: 422.9 types / 1071 tokens
- silicon_bielik_en: 355.6 types / 1328 tokens
- silicon_gemma_en: 331.8 types / 1328 tokens
- real_us: 405.0 types / 1328 tokens

## Embedding homogeneity (mean pairwise cosine; higher = more uniform)
- real_pl: 0.462 (n=150)
- real_pl_all300: 0.455 (n=300)
- silicon_bielik_en: 0.696 (n=150)
- silicon_bielik_pl: 0.827 (n=150)
- silicon_gemma_en: 0.698 (n=150)
- silicon_gemma_pl: 0.703 (n=150)
- real_us: 0.429 (n=150)

## 11. Stop the server and save results to GitHub
Optional: commit `outputs/` from here (needs your git credentials on Alpine), or download the folder from the Jupyter file browser.

In [13]:
if server is not None:
    server.terminate()
    print('Ollama server stopped')
# !cd {ROOT} && git add outputs && git commit -m "Feasibility run outputs" && git push

Ollama server stopped
